In [6]:
from gates_jax import Gate,pauli_vector_to_density_matrix,density_matrix_to_pauli_vector, calculate_steadystate
from QI import purity_cv,partial_trace
import pennylane as qml

ImportError: cannot import name 'Primitive' from 'jax.extend.core' (/Users/eliaszap/anaconda3/envs/pennylane_env/lib/python3.10/site-packages/jax/extend/core.py)

# Testing

First generate random gate sequences:

In [ ]:
def generate_random_gate_sequence(gate_set, depth, num_qubits):
    """Takes list of pennylane gates a circuit depth and qubit number 
    returns a list of sequences (qml.gate, parameter, location) all randomized """
    gates = []
    locations = []
    for d in range(depth):
        gate = np.random.choice(gate_set)
        if gate.num_params > 0:
            params = 2 * np.pi * np.random.rand(gate.num_params)
        else:
            params = None
        qubits = np.random.choice(num_qubits, size=gate.num_wires, replace=False)
        gates.append((gate, params, qubits))
        locations.append(qubits)
    return gates, locations

## Simulation in pennylane 

In [ ]:
def run_random_circuit(gates, num_qubits):
    """Takes a gate sequence and returns the density matrix"""
    dev = qml.device("default.mixed", wires=num_qubits)
    
    @qml.qnode(dev)
    def circuit():
        for i, (gate, params, qubits) in enumerate(gates):
            if params:
                gate(params, wires=qubits)
            else:
                gate( wires=qubits)
        return qml.state()

    circuit()
    return dev._state

# Example usage:
gate_set = [qml.X,qml.Y,qml.CNOT]#[qml.RX, qml.RY, qml.RZ, qml.CNOT]
num_qubits = 2
depth = 3
seq,_ = generate_random_gate_sequence(gate_set, depth, num_qubits)

print(print(run_random_circuit(seq,num_qubits)))
#density_matrix = generate_density_matrix(gate_set, depth, num_qubits)
#print(density_matrix)


[[[[1.+0.j 0.+0.j]
   [0.+0.j 0.+0.j]]

  [[0.+0.j 0.+0.j]
   [0.+0.j 0.+0.j]]]


 [[[0.+0.j 0.+0.j]
   [0.+0.j 0.+0.j]]

  [[0.+0.j 0.+0.j]
   [0.+0.j 0.+0.j]]]]
None


## Code to convert to a gate sequences that can be run by my simulator

In [ ]:
def convert_gate_seq(seq):
    """Converts pennyane gate sequence (qml.gate, parameter, location) into one that is useful for my circuit simulator to ("gatename", parameter, location)"""
    seqnew = []
    for i,s in enumerate(seq): 
        #print(s)
        if s[0].num_wires == 1:
            qubits = [0]
        else:
            qubits = [0,1]
        if s[0].num_params > 0:
            snew = (s[0](wires = qubits,phi = 0).name,s[1],s[2])
        else: 
            snew =  (s[0](wires = qubits).name,s[1],s[2])
        seqnew.append(snew)
    return seqnew

seq,_ = generate_random_gate_sequence(gate_set, depth, num_qubits)
print("\n Pennylane sequence:",seq)
print("\n Converted sequences:", convert_gate_seq(seq))


 Pennylane sequence: [(<class 'pennylane.ops.qubit.non_parametric_ops.PauliY'>, None, array([1])), (<class 'pennylane.ops.op_math.controlled_ops.CNOT'>, None, array([0, 1])), (<class 'pennylane.ops.qubit.non_parametric_ops.PauliX'>, None, array([0]))]

 Converted sequences: [('PauliY', None, array([1])), ('CNOT', None, array([0, 1])), ('PauliX', None, array([0]))]


### run gate sequence in my simulator

In [ ]:
from gates import circuit_matrix 
def run_random_circuit_pauli(gates, num_qubits):
    rho0 = np.zeros((2**num_qubits,2**num_qubits))
    rho0[0,0] = 1 

    convt_seq = convert_gate_seq(gates)

    gate_list = []
    param_list = []
    for s in convt_seq:
        #print(s[2])
        gate = Gate(s[0], Gate_loc= s[2],N_qubits= num_qubits)
        #print(s[2],gate.Gate_loc)
        gate_list.append(gate) 
        if s[1]:
            param_list.append(s[1])

    return pauli_vector_to_density_matrix(circuit_matrix(gate_list,param_list) @ density_matrix_to_pauli_vector(rho0))
print(run_random_circuit_pauli(seq,2) - np.reshape(run_random_circuit(seq, num_qubits),(4,4)))

[[0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]]


# Now can do the comparison 

### Tested gates: 
 - qml.X, qml.Y, qml.Z
 - qml.T, qml.S
 - qml.Hadamard 
 - qml.RX, qml.RY, qml.RZ
 - qml.SWAP
 - qml.IsingXX,qml.IsingYY,qml.IsingZZ
 - CRY, CRX, CRZ

 ### Not tested:
  - channels
  - "XXPlusYY" "XXMinusYY"
 ### Broken gates:
  - None right now


In [ ]:
#Tested 
gate_set = [ qml.Hadamard,qml.S,qml.CRY,qml.CRZ] 
depth = 3
num_qubits = 4
n_runs = 5

#make list of gate sequences
list_gatesequences = [] 
for i in range(n_runs):
    list_gatesequences.append(generate_random_gate_sequence(gate_set,depth,num_qubits)[0])

#simulate in two different ways and compare the result
list_normdifference = []
for seq in list_gatesequences:
    #print(seq)
    list_normdifference.append(np.linalg.norm(run_random_circuit_pauli(seq,num_qubits) - np.reshape(run_random_circuit(seq, num_qubits),(2**num_qubits,2**num_qubits))))
print("\n Max norm difference", np.max(list_normdifference))
print("list",list_normdifference)


 Max norm difference 4.577566798522237e-16
list [0.0, 4.577566798522237e-16, 1.2643266404855218e-16, 3.3537189618722464e-16, 2.486410087150353e-16]
